# 31_1: Earnings-Date Confounding Check

Does the headline event-study result (`31_event_study.ipynb`, reused via
`31_0_event_study_descriptives.ipynb`'s regression guard) survive once events whose
[-1,+5] reaction window overlaps a scheduled earnings announcement are excluded?

This notebook reuses, but never modifies, the existing engine:
- `src/event_study_utils.py` for `event_window_stats`, `bmp_test`,
  `cluster_robust_mean_test_fast`, `get_trading_day_offset`, and the estimation-window
  constants.
- The retained event table and `yf_ticker`/`TICKER_SUFFIX_MAP` convention from
  `31_0_event_study_descriptives.ipynb` (copied here, since that
  mapping is not itself stored in the saved parquet).

New logic specific to earnings dates (fetching, exchange-timezone localization, reaction-day
classification, coverage classification) lives in `src/earnings_dates.py`, written for this
check.

**Day-0 anchor convention:** `anchor='ffill'` throughout (previous-or-equal trading day),
matching `event_window_stats`'s default and every anchor choice in
`31_0_event_study_descriptives.ipynb`.

All outputs are written to `results/phase_iv/` with an `earn_` prefix. No existing notebook
or `assump_v2_*` / `*_v3` output is read for writing, or modified.

In [1]:
import sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..').resolve()))
from src.event_study_utils import (
    event_window_stats, bmp_test, cluster_robust_mean_test_fast,
    get_trading_day_offset, ESTIMATION_START, ESTIMATION_END, MIN_EST_OBS,
)
from src.earnings_dates import (
    fetch_raw_earnings_dates, get_exchange_timezone, localize_to_exchange,
    classify_reaction_day, nearest_gap_days, classify_coverage,
    EARNINGS_FETCH_LIMIT, COVERAGE_GAP_DAYS,
)

OUTPUT_DIR = Path('results/phase_iv')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RNG_SEED = 42
ANCHOR = 'ffill'          # day-0 anchor convention used throughout this notebook
EVENT_WINDOW = (-1, 5)    # the main event-study window, for earn_in_window / flag comparisons
PM5_WINDOW = (-5, 5)      # Nicolas et al. (2023) earnings-confounding rule
SENSITIVITY_WINDOWS = [(-1, 0), (-1, 1), (-1, 5)]
TODAY = pd.Timestamp('2026-10-29')

T_START = time.time()
print(f'Anchor convention: {ANCHOR!r}   Main event window: {EVENT_WINDOW}   Today: {TODAY.date()}')

Anchor convention: 'ffill'   Main event window: (-1, 5)   Today: 2026-10-29


## Step 0 — Load the retained event sample (reused, not recomputed)

Same four inputs and the same `TICKER_SUFFIX_MAP` as
`31_0_event_study_descriptives.ipynb`, so that `yf_ticker` matches exactly. The map itself
is not persisted in `event_study_results_v3.parquet`, so it is reproduced verbatim here
(copied from the v2 notebook) rather than re-derived.

In [2]:
# Exchange suffix map -- copied verbatim from 31_event_study.ipynb / 31_0_..._v2.ipynb
# so yf_ticker reconstructs identically. Not stored in event_study_results_v3.parquet.
TICKER_SUFFIX_MAP = {
    # Germany (Frankfurt)
    'SIE':  'SIE.DE',  'BAYN': 'BAYN.DE', 'LHA':  'LHA.DE',
    'BAS':  'BAS.DE',  'ADS':  'ADS.DE',  'VOW3': 'VOW3.DE', 'PAH3': 'PAH3.DE',
    # Switzerland
    'ZURN': 'ZURN.SW', 'NESN': 'NESN.SW', 'NOVN': 'NOVN.SW',
    # UK (London)
    'HSBA': 'HSBA.L',  'LLOY': 'LLOY.L',  'GLEN': 'GLEN.L',
    'RR.':  'RR.L',    'BA.':  'BA.L',     'SVT':  'SVT.L',
    'ULVR': 'ULVR.L',  'UU.':  'UU.L',     'NG.':  'NG.L',
    'AV.':  'AV.L',    'LSEG': 'LSEG.L',   'STAN': 'STAN.L',
    # France (Paris)
    'RNO':  'RNO.PA',  'PUB':  'PUB.PA',   'AC':   'AC.PA',
    'ML':   'ML.PA',   'ALO':  'ALO.PA',   'VIE':  'VIE.PA',
    # Netherlands (Amsterdam)
    'PHIA': 'PHIA.AS', 'UMG':  'UMG.AS',
    # Sweden (Stockholm) -- B-share tickers use hyphen on yfinance
    'SAAB B':  'SAAB-B.ST', 'VOLV B':   'VOLV-B.ST', 'ERIC B': 'ERIC-B.ST',
    # Denmark (Copenhagen)
    'PNDORA':  'PNDORA.CO', 'MAERSK B': 'MAERSK-B.CO', 'VWS': 'VWS.CO',
    # Finland
    'NOKIA': 'NOKIA.HE',
    # Spain
    'CLNX':  'CLNX.MC',
    # Belgium
    'ABI':   'ABI.BR',
    # Japan (Tokyo)
    '7203': '7203.T', '8604': '8604.T', '6758': '6758.T',
    '7751': '7751.T', '7733': '7733.T', '7267': '7267.T',
    '7974': '7974.T', '9201': '9201.T', '8306': '8306.T',
    '7201': '7201.T', '9434': '9434.T', '6501': '6501.T',
    '7269': '7269.T', '8316': '8316.T', '2587': '2587.T',
    # Hong Kong
    '823':  '0823.HK',
    # Australia (ASX)
    'MQG': 'MQG.AX', 'QAN': 'QAN.AX', 'WOW': 'WOW.AX', 'WBC': 'WBC.AX',
    # Canada (Toronto)
    'FFH': 'FFH.TO',
    # US -- special case (B-share uses hyphen)
    'BRKB': 'BRK-B',
}

In [3]:
PRICE_CACHE = OUTPUT_DIR / 'price_cache_v4_acwi.parquet'
EVENT_STUDY_RESULTS = OUTPUT_DIR / 'event_study_results_v3.parquet'
MSCI_PATH = Path('data/00_raw/msci_world.csv')

prices = pd.read_parquet(PRICE_CACHE)
prices.index = pd.to_datetime(prices.index)
returns = np.log(prices / prices.shift(1)).dropna(how='all')

event_df = pd.read_parquet(EVENT_STUDY_RESULTS)
event_df['event_date'] = pd.to_datetime(event_df['event_date'])
event_df['yf_ticker'] = event_df['ticker'].map(TICKER_SUFFIX_MAP).fillna(event_df['ticker'])

msci_df = pd.read_csv(MSCI_PATH, sep=';', skiprows=7)
msci_df = msci_df.drop_duplicates(subset='Ticker', keep='first')
event_df = event_df.merge(
    msci_df[['Ticker', 'Location']].rename(columns={'Ticker': 'ticker_msci', 'Location': 'region'}),
    left_on='ticker', right_on='ticker_msci', how='left'
).drop(columns=['ticker_msci'])
event_df['is_us'] = event_df['region'] == 'United States'

mkt_rets = returns['ACWI'].dropna()
trading_cal = mkt_rets.index

firm_rets_cache = {
    tkr: (returns[tkr].dropna() if tkr in returns.columns else None)
    for tkr in event_df['yf_ticker'].unique()
}

print(f'Retained events : {len(event_df):,} (from {event_df["ticker"].nunique()} firms, '
      f'{event_df["yf_ticker"].nunique()} unique yf_tickers)')
print(f'US events       : {event_df["is_us"].sum():,}   Non-US: {(~event_df["is_us"]).sum():,}')

Retained events : 1,463 (from 210 firms, 210 unique yf_tickers)
US events       : 1,061   Non-US: 402


## Regression guard

Must reproduce `event_study_results_v3.parquet`'s full-sample mean CAR[-1,+5] to within 1e-9
before anything below runs. This mirrors the guard in `31_0_event_study_assumptions_v2.ipynb`
(which validated per-event `car`/`scar`); here it is applied to the aggregate mean CAR, since
that is the single number this notebook's sensitivity table (Step 5) will be compared against.

In [4]:
V3_MEAN_CAR_FULL_SAMPLE = -0.004627887563014905  # from event_study_summary_v3.csv, label='ALL PEAKS'

mean_car_check = event_df['car'].mean()
diff = abs(mean_car_check - V3_MEAN_CAR_FULL_SAMPLE)
print(f'Regression guard: recomputed mean CAR[-1,+5] = {mean_car_check:.12f}')
print(f'                  v3 summary mean CAR[-1,+5] = {V3_MEAN_CAR_FULL_SAMPLE:.12f}')
print(f'                  abs diff                    = {diff:.3e}')

if diff >= 1e-9:
    raise AssertionError(
        f'Regression guard FAILED: recomputed mean CAR ({mean_car_check}) differs from the '
        f'v3 notebook value ({V3_MEAN_CAR_FULL_SAMPLE}) by {diff:.3e} (>= 1e-9). Halting -- '
        'do not trust anything below until this is resolved.'
    )
print('REGRESSION GUARD PASSED.')

Regression guard: recomputed mean CAR[-1,+5] = -0.004627887563
                  v3 summary mean CAR[-1,+5] = -0.004627887563
                  abs diff                    = 0.000e+00
REGRESSION GUARD PASSED.


## Step 1 — Fetch earnings dates

`yfinance.Ticker.get_earnings_dates()` caps `limit` at 100 (not the default 12, which covers
roughly two years) -- used here for every ticker. Raw results are cached to `earn_raw.csv` so
reruns do not refetch.

**Timezone check, done explicitly rather than assumed:** the returned `DatetimeIndex` is
tz-aware, but it is labeled `America/New_York` for *every* ticker regardless of its actual
listing exchange -- confirmed below for a US, a German and a Japanese ticker. It is **not**
already in exchange-local time; Step 2 converts it using each ticker's real exchange timezone
(`Ticker.fast_info.timezone`).

In [5]:
_check_tickers = ['AAPL', 'SIE.DE', '7203.T']
print('Spot-check: yfinance earnings-date index timezone label, by ticker')
for t in _check_tickers:
    import yfinance as yf
    idx = yf.Ticker(t).get_earnings_dates(limit=5).index
    print(f'  {t:10s} tz label = {idx.tz}  (sample: {idx[0]})')
print()
print('All three are labeled America/New_York regardless of exchange -- confirms the '
      'timestamps are NOT pre-converted to exchange-local time. Step 2 converts explicitly.')

Spot-check: yfinance earnings-date index timezone label, by ticker


  AAPL       tz label = America/New_York  (sample: 2026-11-02 15:00:00-05:00)


  SIE.DE     tz label = America/New_York  (sample: 2026-11-12 11:00:00-05:00)


  7203.T     tz label = America/New_York  (sample: 2026-11-05 01:00:00-05:00)

All three are labeled America/New_York regardless of exchange -- confirms the timestamps are NOT pre-converted to exchange-local time. Step 2 converts explicitly.


In [6]:
RAW_CACHE_PATH = OUTPUT_DIR / 'earn_raw.csv'
TZ_CACHE_PATH = OUTPUT_DIR / 'earn_exchange_tz.csv'

all_yf_tickers = sorted(event_df['yf_ticker'].unique().tolist())

if RAW_CACHE_PATH.exists():
    raw_df = pd.read_csv(RAW_CACHE_PATH)
    raw_df['earnings_date_raw'] = pd.to_datetime(raw_df['earnings_date_raw'], utc=True)
    print(f'Loaded cached raw earnings dates: {raw_df.shape} from {RAW_CACHE_PATH}')
else:
    print(f'Fetching earnings dates for {len(all_yf_tickers)} tickers (limit={EARNINGS_FETCH_LIMIT})...')
    t0 = time.time()
    raw_df, fetch_log_df = fetch_raw_earnings_dates(all_yf_tickers)
    print(f'Fetch complete in {time.time()-t0:.1f}s. raw_df shape={raw_df.shape}')
    raw_df.to_csv(RAW_CACHE_PATH, index=False)
    fetch_log_df.to_csv(OUTPUT_DIR / 'earn_fetch_log.csv', index=False)

if TZ_CACHE_PATH.exists():
    exchange_tz = pd.read_csv(TZ_CACHE_PATH, index_col=0).iloc[:, 0].to_dict()
    print(f'Loaded cached exchange timezones for {len(exchange_tz)} tickers from {TZ_CACHE_PATH}')
else:
    print('Fetching exchange timezone per ticker...')
    exchange_tz = {}
    for tkr in all_yf_tickers:
        exchange_tz[tkr] = get_exchange_timezone(tkr)
        time.sleep(0.2)
    pd.Series(exchange_tz).to_csv(TZ_CACHE_PATH)

n_missing_tz = sum(v is None or (isinstance(v, float) and pd.isna(v)) for v in exchange_tz.values())
print(f'Tickers with no resolvable exchange timezone: {n_missing_tz} / {len(exchange_tz)}')

Loaded cached raw earnings dates: (15748, 5) from results/phase_iv/earn_raw.csv
Loaded cached exchange timezones for 210 tickers from results/phase_iv/earn_exchange_tz.csv
Tickers with no resolvable exchange timezone: 5 / 210


In [7]:
fetch_log_path = OUTPUT_DIR / 'earn_fetch_log.csv'
if fetch_log_path.exists():
    fetch_log_df = pd.read_csv(fetch_log_path)
    print('Per-ticker fetch log (n dates, earliest, latest, any error):')
    print(fetch_log_df.to_string(index=False))
    n_failed = fetch_log_df['error'].notna().sum()
    print(f'\n{n_failed} / {len(fetch_log_df)} tickers returned no earnings data '
          f'(these become coverage_status="no_data" in Step 3).')

In [8]:
n_before = len(raw_df)
raw_df['earnings_date_utc_naive'] = raw_df['earnings_date_raw'].dt.tz_convert('UTC').dt.tz_localize(None)

# Keep only dates up to today.
raw_df = raw_df[raw_df['earnings_date_utc_naive'] <= TODAY].copy()
n_after_date_filter = len(raw_df)

# Drop estimated rows: an earnings date with no Reported EPS has not actually been reported
# yet (yfinance still lists the next few scheduled calls with only an EPS *estimate*) and is
# therefore not a confirmed historical announcement date.
raw_df = raw_df[raw_df['reported_eps'].notna()].copy()
n_after_estimated_filter = len(raw_df)

print(f'Raw rows                         : {n_before:,}')
print(f'After dropping future (> today)  : {n_after_date_filter:,}')
print(f'After dropping estimated/unreported: {n_after_estimated_filter:,}')

Raw rows                         : 15,748
After dropping future (> today)  : 15,555
After dropping estimated/unreported: 15,447


## Step 2 — Reaction day

Convert each remaining announcement timestamp to its exchange's local time
(`Ticker.fast_info.timezone`, resolved per ticker in Step 1), then classify which trading
day(s) the market reaction should land on:

- before 16:00 local → reaction day = that trading day
- at/after 16:00 local → reaction day = the next trading day
- midnight (00:00:00, i.e. no real time component) or missing → `timing_unknown`; both that
  day and the next trading day are marked as possibly affected

In [9]:
raw_df['exchange_tz'] = raw_df['yf_ticker'].map(exchange_tz)
n_no_tz = raw_df['exchange_tz'].isna().sum()
print(f'Rows with no resolvable exchange timezone (local_ts -> NaT, timing_unknown): {n_no_tz}')

raw_df['local_ts'] = raw_df.apply(
    lambda r: localize_to_exchange(r['earnings_date_raw'], r['exchange_tz']), axis=1)

_reaction = raw_df['local_ts'].apply(lambda ts: classify_reaction_day(ts, trading_cal))
raw_df['reaction_days'] = _reaction.apply(lambda x: x[0])
raw_df['timing_unknown'] = _reaction.apply(lambda x: x[1])

print(f'Earnings rows with timing_unknown=True: {raw_df["timing_unknown"].sum():,} '
      f'({raw_df["timing_unknown"].mean()*100:.1f}%)')

# One row per (yf_ticker, reaction_day), for the pm5 / in_window flags in Step 4, and one
# calendar-date-per-ticker index (the announcement's own local date) for the Step 3 coverage
# gap calculation -- these are deliberately different: coverage asks "is there earnings data
# near this event at all", flags ask "does a specific trading day fall in the event window".
_rd_rows = [(r['yf_ticker'], d) for _, r in raw_df.iterrows() for d in r['reaction_days']]
reaction_df = pd.DataFrame(_rd_rows, columns=['yf_ticker', 'reaction_day'])
reaction_by_ticker = {
    tkr: pd.DatetimeIndex(sorted(g['reaction_day'].unique()))
    for tkr, g in reaction_df.groupby('yf_ticker')
}

raw_df['local_date'] = raw_df['local_ts'].apply(lambda ts: pd.Timestamp(ts.date()) if pd.notna(ts) else pd.NaT)
earn_cal_dates_by_ticker = {
    tkr: pd.DatetimeIndex(sorted(g['local_date'].dropna().unique()))
    for tkr, g in raw_df.groupby('yf_ticker')
}
tickers_with_data = {t for t, d in earn_cal_dates_by_ticker.items() if len(d) > 0}
print(f'Tickers with at least one usable earnings date: {len(tickers_with_data)} / {len(all_yf_tickers)}')

Rows with no resolvable exchange timezone (local_ts -> NaT, timing_unknown): 0


Earnings rows with timing_unknown=True: 2,944 (19.1%)


Tickers with at least one usable earnings date: 195 / 210


## Step 3 — Coverage report (before any flag)

An event is:
- **`covered`** if the nearest earnings date before it, and the nearest after it, are each
  within 120 calendar days (a larger gap suggests a missing quarter in the fetched history).
- **`gap`** if the ticker has earnings data but one or both sides exceed that gap (or are
  missing).
- **`no_data`** if the ticker returned no usable earnings dates at all.

In [10]:
coverage_rows = []
for _, ev in event_df.iterrows():
    tkr = ev['yf_ticker']
    has_data = tkr in tickers_with_data
    dates = earn_cal_dates_by_ticker.get(tkr, pd.DatetimeIndex([]))
    days_before, days_after = nearest_gap_days(ev['event_date'], dates) if has_data else (np.nan, np.nan)
    status = classify_coverage(days_before, days_after, has_data)
    coverage_rows.append(dict(
        peak_id=ev['peak_id'], ticker=ev['ticker'], yf_ticker=tkr, is_us=ev['is_us'],
        region=ev['region'], year=ev['event_date'].year, event_date=ev['event_date'],
        days_before=days_before, days_after=days_after, coverage_status=status,
    ))

coverage_df = pd.DataFrame(coverage_rows)
event_df = event_df.merge(coverage_df[['peak_id', 'coverage_status', 'days_before', 'days_after']],
                           on='peak_id', how='left')

print('Coverage status, overall:')
print(coverage_df['coverage_status'].value_counts())
print()
print('Coverage status, share (%):')
print((coverage_df['coverage_status'].value_counts(normalize=True) * 100).round(1))

Coverage status, overall:
coverage_status
covered    1295
gap         115
no_data      53
Name: count, dtype: int64

Coverage status, share (%):
coverage_status
covered    88.5
gap         7.9
no_data     3.6
Name: proportion, dtype: float64


In [11]:
print('Coverage status by US vs non-US (row %):')
print((coverage_df.groupby('is_us')['coverage_status']
       .value_counts(normalize=True).unstack() * 100).round(1))
print()
print('Coverage status by year (row %):')
by_year = (coverage_df.groupby('year')['coverage_status']
           .value_counts(normalize=True).unstack().fillna(0) * 100).round(1)
print(by_year)

coverage_df.to_csv(OUTPUT_DIR / 'earn_coverage.csv', index=False)
print(f'\nSaved to {OUTPUT_DIR / "earn_coverage.csv"}')

Coverage status by US vs non-US (row %):
coverage_status  covered   gap  no_data
is_us                                  
False               67.9  20.9     11.2
True                96.3   2.9      0.8

Coverage status by year (row %):
coverage_status  covered   gap  no_data
year                                   
2009                80.3  15.8      3.9
2010                77.4  20.4      2.2
2011                90.2   7.6      2.2
2012                89.7   6.2      4.1
2013                88.7   8.5      2.8
2014                93.1   2.8      4.2
2015                78.9  17.9      3.3
2016                88.5   8.3      3.1
2017                92.7   3.1      4.2
2018                94.1   1.0      4.9
2019                86.7  10.8      2.4
2020                95.1   2.4      2.4
2021                90.4   6.8      2.7
2022                93.7   3.8      2.5
2023                90.8   1.5      7.7
2024                85.7   7.1      7.1

Saved to results/phase_iv/earn_coverage.csv


In [12]:
print('Earliest event date per ticker vs. earliest fetched earnings date '
      '(tickers with data, first 20 alphabetically):')
for tkr in sorted(tickers_with_data)[:20]:
    ev_earliest = event_df.loc[event_df['yf_ticker'] == tkr, 'event_date'].min()
    earn_earliest = earn_cal_dates_by_ticker[tkr].min()
    flag = '  <-- event predates earliest fetched earnings date' if ev_earliest < earn_earliest else ''
    print(f'  {tkr:15s} event={ev_earliest.date()}   earnings={earn_earliest.date()}{flag}')

Earliest event date per ticker vs. earliest fetched earnings date (tickers with data, first 20 alphabetically):
  6501.T          event=2018-05-25   earnings=2004-07-29
  6758.T          event=2009-06-04   earnings=2004-07-28
  7201.T          event=2010-05-26   earnings=2004-07-29
  7203.T          event=2009-03-26   earnings=2004-08-03
  7267.T          event=2014-06-15   earnings=2004-07-28
  7733.T          event=2011-10-17   earnings=2004-11-08
  7751.T          event=2019-06-18   earnings=2004-07-28
  7974.T          event=2011-07-28   earnings=2013-07-31  <-- event predates earliest fetched earnings date
  8306.T          event=2010-04-21   earnings=2010-02-03
  8316.T          event=2014-08-07   earnings=2009-01-28
  8604.T          event=2012-10-29   earnings=2004-07-29
  9434.T          event=2020-04-13   earnings=2019-02-05
  AAL             event=2012-03-01   earnings=2014-01-28  <-- event predates earliest fetched earnings date
  ABNB            event=2022-11-07   earnings

## Step 4 — Flags

Computed **only** for `covered` events. For `gap` and `no_data` events the flag is `NaN`
(never `False`) -- a missing earnings calendar must not silently read as "no confound".

- `earn_pm5`: any reaction day within [-5,+5] trading days of day 0 (Nicolas et al. 2023).
- `earn_in_window`: any reaction day inside the event's own CAR window ([-1,+5]).

In [13]:
def compute_flag(ev, window):
    tkr = ev['yf_ticker']
    rdays = reaction_by_ticker.get(tkr, pd.DatetimeIndex([]))
    win_s = get_trading_day_offset(trading_cal, ev['event_date'], window[0], anchor=ANCHOR)
    win_e = get_trading_day_offset(trading_cal, ev['event_date'], window[1], anchor=ANCHOR)
    if win_s is None or win_e is None:
        return False
    return bool(((rdays >= win_s) & (rdays <= win_e)).any())

pm5_flags, in_window_flags = [], []
for _, ev in event_df.iterrows():
    if ev['coverage_status'] != 'covered':
        pm5_flags.append(np.nan)
        in_window_flags.append(np.nan)
        continue
    pm5_flags.append(compute_flag(ev, PM5_WINDOW))
    in_window_flags.append(compute_flag(ev, EVENT_WINDOW))

event_df['earn_pm5'] = pm5_flags
event_df['earn_in_window'] = in_window_flags

print('earn_pm5:')
print(event_df['earn_pm5'].value_counts(dropna=False))
print('\nearn_in_window:')
print(event_df['earn_in_window'].value_counts(dropna=False))

n_covered = (event_df['coverage_status'] == 'covered').sum()
print(f'\nShare of covered events flagged earn_pm5=True      : '
      f'{event_df["earn_pm5"].sum() / n_covered * 100:.1f}%')
print(f'Share of covered events flagged earn_in_window=True: '
      f'{event_df["earn_in_window"].sum() / n_covered * 100:.1f}%')

earn_pm5:
earn_pm5
False    982
True     313
NaN      168
Name: count, dtype: int64

earn_in_window:
earn_in_window
False    1065
True      230
NaN       168
Name: count, dtype: int64

Share of covered events flagged earn_pm5=True      : 24.2%
Share of covered events flagged earn_in_window=True: 17.8%


In [14]:
# Guard: no NaN-status (gap / no_data) event may have gotten a False flag -- NaN must stay NaN.
_non_covered = event_df['coverage_status'] != 'covered'
assert event_df.loc[_non_covered, 'earn_pm5'].isna().all(), \
    'FAIL: a non-covered event has a non-NaN earn_pm5 (should be NaN, never False)'
assert event_df.loc[_non_covered, 'earn_in_window'].isna().all(), \
    'FAIL: a non-covered event has a non-NaN earn_in_window (should be NaN, never False)'
assert event_df.loc[~_non_covered, 'earn_pm5'].isin([True, False]).all(), \
    'FAIL: a covered event has a non-boolean earn_pm5'
print('GUARD PASSED: every gap/no_data event has NaN flags; every covered event has a boolean flag.')

GUARD PASSED: every gap/no_data event has NaN flags; every covered event has a boolean flag.


## Step 5 — Sensitivity

For each window, report n / mean CAR / BMP p / firm-clustered p / Wilcoxon p for:

- **(a) all events** -- the full retained sample (for reference only).
- **(b) covered events** -- sample restricted to events with adequate earnings-calendar
  coverage, confounding not yet removed.
- **(c) covered & `earn_pm5`==False** -- covered events with no earnings reaction nearby.
- **(d) covered & `earn_pm5`==True** -- covered events with an earnings reaction nearby.

**The valid comparison is (b) vs. (c)**: both draw from the same covered sample, so any
difference reflects removing earnings confounding, not a change in which events are included.
(a) vs. (c) is not a valid test of confounding on its own, since it also changes the sample
(adds gap/no_data events absent from (c)).

In [15]:
def window_stats_for(window, sub_df):
    a, b = window
    cars, scars, tickers = [], [], []
    for _, ev in sub_df.iterrows():
        firm_rets = firm_rets_cache.get(ev['yf_ticker'])
        if firm_rets is None:
            continue
        res = event_window_stats(firm_rets, mkt_rets, trading_cal, ev['event_date'], a, b, anchor=ANCHOR)
        if res is None:
            continue
        cars.append(res['car']); scars.append(res['scar']); tickers.append(ev['ticker'])
    cars, scars, tickers = np.array(cars), np.array(scars), np.array(tickers)
    n = len(cars)
    if n == 0:
        return dict(n=0, mean_car=np.nan, bmp_p=np.nan, clustered_p=np.nan, wilcoxon_p=np.nan)
    _, bmp_p = bmp_test(scars)
    clu = cluster_robust_mean_test_fast(cars, tickers)
    wil_p = stats.wilcoxon(cars, alternative='less')[1]
    return dict(n=n, mean_car=np.mean(cars), bmp_p=bmp_p, clustered_p=clu['p_value'], wilcoxon_p=wil_p)

covered_df = event_df[event_df['coverage_status'] == 'covered']
samples = {
    'a_all': event_df,
    'b_covered': covered_df,
    'c_covered_clean': covered_df[covered_df['earn_pm5'] == False],
    'd_covered_confounded': covered_df[covered_df['earn_pm5'] == True],
}

sensitivity_rows = []
for window in SENSITIVITY_WINDOWS:
    for label, sub in samples.items():
        r = window_stats_for(window, sub)
        r.update(window=f'[{window[0]:+d},{window[1]:+d}]', sample=label)
        sensitivity_rows.append(r)

sensitivity_df = pd.DataFrame(sensitivity_rows)[
    ['window', 'sample', 'n', 'mean_car', 'bmp_p', 'clustered_p', 'wilcoxon_p']]
print(sensitivity_df.to_string(index=False))
sensitivity_df.to_csv(OUTPUT_DIR / 'earn_sensitivity.csv', index=False)

 window               sample    n  mean_car    bmp_p  clustered_p  wilcoxon_p
[-1,+0]                a_all 1463 -0.002090 0.019388     0.013362    0.020168
[-1,+0]            b_covered 1295 -0.001941 0.014146     0.025474    0.036066
[-1,+0]      c_covered_clean  982 -0.001230 0.020288     0.146500    0.087559
[-1,+0] d_covered_confounded  313 -0.004171 0.176958     0.065133    0.107886
[-1,+1]                a_all 1463 -0.002504 0.007372     0.007834    0.044592
[-1,+1]            b_covered 1295 -0.002288 0.007408     0.016191    0.090440
[-1,+1]      c_covered_clean  982 -0.000881 0.031924     0.222736    0.179299
[-1,+1] d_covered_confounded  313 -0.006701 0.057055     0.026098    0.113665
[-1,+5]                a_all 1463 -0.004628 0.000100     0.000699    0.001178
[-1,+5]            b_covered 1295 -0.004353 0.000319     0.002818    0.006093
[-1,+5]      c_covered_clean  982 -0.001346 0.017060     0.259442    0.199736
[-1,+5] d_covered_confounded  313 -0.013785 0.002263     0.00070

In [16]:
covered_reg_df = covered_df.dropna(subset=['earn_pm5']).copy()
covered_reg_df['earn_pm5'] = covered_reg_df['earn_pm5'].astype(bool)
earn_reg = smf.ols('car ~ earn_pm5', data=covered_reg_df).fit(
    cov_type='cluster', cov_kwds={'groups': covered_reg_df['ticker']})
print('Firm-clustered regression of CAR[-1,+5] on earn_pm5, among covered events:')
print(earn_reg.summary2().tables[1])

earn_reg_coef = earn_reg.params.get('earn_pm5[T.True]', np.nan)
earn_reg_p = earn_reg.pvalues.get('earn_pm5[T.True]', np.nan)

pd.DataFrame([dict(
    n_covered=len(covered_reg_df), reg_coef_earn_pm5=earn_reg_coef, reg_p_earn_pm5=earn_reg_p,
)]).to_csv(OUTPUT_DIR / 'earn_regression.csv', index=False)

Firm-clustered regression of CAR[-1,+5] on earn_pm5, among covered events:
                     Coef.  Std.Err.         z     P>|z|    [0.025    0.975]
Intercept        -0.001346  0.002083 -0.646260  0.518111 -0.005428  0.002736
earn_pm5[T.True] -0.012439  0.005271 -2.360107  0.018270 -0.022770 -0.002109


In [17]:
_main_row = sensitivity_df[(sensitivity_df['window'] == '[-1,+5]')].set_index('sample')
_b, _c, _d = _main_row.loc['b_covered'], _main_row.loc['c_covered_clean'], _main_row.loc['d_covered_confounded']
_pct_pm5 = covered_df['earn_pm5'].mean() * 100
_pct_covered = (event_df['coverage_status'] == 'covered').mean() * 100

_summary = (
    f"Of {len(event_df):,} retained events, {_pct_covered:.1f}% have adequate earnings-calendar "
    f"coverage (both neighboring earnings dates within {COVERAGE_GAP_DAYS} days); among those "
    f"covered events, {_pct_pm5:.1f}% have an earnings reaction day within +/-5 trading days of "
    f"day 0 (earn_pm5). On the main window [-1,+5], "
    f"mean CAR moves from {_b['mean_car']*100:.3f}% (covered, n={_b['n']:.0f}, BMP p={_b['bmp_p']:.4f}) "
    f"to {_c['mean_car']*100:.3f}% (covered & earn_pm5=False, n={_c['n']:.0f}, BMP p={_c['bmp_p']:.4f}) "
    f"once earnings-confounded events are excluded -- the valid comparison for isolating the "
    f"effect of earnings confounding. The excluded (earn_pm5=True) subgroup alone shows mean "
    f"CAR={_d['mean_car']*100:.3f}% (n={_d['n']:.0f}, BMP p={_d['bmp_p']:.4f}). The firm-clustered "
    f"regression of CAR on earn_pm5 among covered events gives a coefficient of "
    f"{earn_reg_coef*100:+.3f} percentage points (p={earn_reg_p:.4f})."
)
print(_summary)

with open(OUTPUT_DIR / 'earn_summary.txt', 'w') as f:
    f.write(_summary + '\n')

Of 1,463 retained events, 88.5% have adequate earnings-calendar coverage (both neighboring earnings dates within 120 days); among those covered events, 24.2% have an earnings reaction day within +/-5 trading days of day 0 (earn_pm5). On the main window [-1,+5], mean CAR moves from -0.435% (covered, n=1295, BMP p=0.0003) to -0.135% (covered & earn_pm5=False, n=982, BMP p=0.0171) once earnings-confounded events are excluded -- the valid comparison for isolating the effect of earnings confounding. The excluded (earn_pm5=True) subgroup alone shows mean CAR=-1.379% (n=313, BMP p=0.0023). The firm-clustered regression of CAR on earn_pm5 among covered events gives a coefficient of -1.244 percentage points (p=0.0183).


**Interpretation**: 

* Of 1,463 retained events, 88.5% have adequate earnings-calendar coverage (both neighboring earnings dates within 120 days); among those covered events, 24.2% have an earnings reaction day within +/-5 trading days of day 0 (earn_pm5). 

* On the main window [-1,+5], mean CAR moves from -0.435% (covered, n=1295, BMP p=0.0003) to -0.135% (covered & earn_pm5=False, n=982, BMP p=0.0171) once earnings-confounded events are excluded -- the valid comparison for isolating the effect of earnings confounding. 

* The excluded (earn_pm5=True) subgroup alone shows mean CAR=-1.379% (n=313, BMP p=0.0023). The firm-clustered regression of CAR on earn_pm5 among covered events gives a coefficient of -1.244 percentage points (p=0.0183).

## Spot-check: random US events with matched earnings dates

For manual verification against SEC 8-K filings. Seeded (42) so this is reproducible, not
cherry-picked.

In [18]:
rng = np.random.default_rng(RNG_SEED)
us_events = event_df[event_df['is_us']].reset_index(drop=True)
sample_idx = rng.choice(len(us_events), size=10, replace=False)
spot_check_sample = us_events.iloc[sample_idx]

spot_check_rows = []
for _, ev in spot_check_sample.iterrows():
    dates = earn_cal_dates_by_ticker.get(ev['yf_ticker'], pd.DatetimeIndex([]))
    before = dates[dates < ev['event_date']]
    after = dates[dates > ev['event_date']]
    nearest_before = before.max() if len(before) else pd.NaT
    nearest_after = after.min() if len(after) else pd.NaT
    row = dict(ticker=ev['ticker'], yf_ticker=ev['yf_ticker'], event_date=ev['event_date'].date(),
               nearest_earnings_before=nearest_before.date() if pd.notna(nearest_before) else None,
               nearest_earnings_after=nearest_after.date() if pd.notna(nearest_after) else None,
               coverage_status=ev['coverage_status'], earn_pm5=ev['earn_pm5'])
    spot_check_rows.append(row)
    print(f"{row['ticker']:8s} event={row['event_date']}  "
          f"earnings before={row['nearest_earnings_before']}  after={row['nearest_earnings_after']}  "
          f"coverage={row['coverage_status']}  earn_pm5={row['earn_pm5']}")

pd.DataFrame(spot_check_rows).to_csv(OUTPUT_DIR / 'earn_spot_check.csv', index=False)

AMZN     event=2022-10-05  earnings before=2022-07-28  after=2022-10-27  coverage=covered  earn_pm5=False
MU       event=2018-06-22  earnings before=2018-06-20  after=2018-09-20  coverage=covered  earn_pm5=True
AMZN     event=2023-11-20  earnings before=2023-10-26  after=2024-02-01  coverage=covered  earn_pm5=False
META     event=2017-09-13  earnings before=2017-07-26  after=2017-11-01  coverage=covered  earn_pm5=False
GOOG     event=2017-11-29  earnings before=2017-10-26  after=2018-02-01  coverage=covered  earn_pm5=False
GOOG     event=2017-03-06  earnings before=2017-01-26  after=2017-04-27  coverage=covered  earn_pm5=False
META     event=2022-12-14  earnings before=2022-10-26  after=2023-02-01  coverage=covered  earn_pm5=False
AVGO     event=2017-11-13  earnings before=2017-08-24  after=2017-12-06  coverage=covered  earn_pm5=False
CAT      event=2018-10-23  earnings before=2018-07-30  after=2019-01-28  coverage=covered  earn_pm5=True
SBUX     event=2022-06-22  earnings before=2022-

## Guards and housekeeping

- No credentials in this module -- `yfinance` calls are unauthenticated.
- `.gitignore` already excludes `results/` wholesale (and `data/`), so the raw earnings cache
  (`earn_raw.csv`) is not committed -- confirmed below rather than assumed.

In [19]:
gitignore_path = Path('../.gitignore')
gitignore_text = gitignore_path.read_text() if gitignore_path.exists() else ''
results_ignored = any(line.strip().rstrip('/') == 'results' for line in gitignore_text.splitlines())
print(f'.gitignore excludes results/ : {results_ignored}')
if not results_ignored:
    print('WARNING: results/ is not ignored -- earn_raw.csv would be committed. Not committing it here.')

raw_cache_size_mb = RAW_CACHE_PATH.stat().st_size / 1e6 if RAW_CACHE_PATH.exists() else 0
print(f'earn_raw.csv size: {raw_cache_size_mb:.2f} MB')

.gitignore excludes results/ : True
earn_raw.csv size: 0.73 MB


## Runtime

In [20]:
print(f'TOTAL NOTEBOOK RUNTIME (excluding any first-run fetch, which is cached above): '
      f'{time.time() - T_START:.1f}s')

TOTAL NOTEBOOK RUNTIME (excluding any first-run fetch, which is cached above): 43.3s
